In [4]:
import re

# ============================================================
# Q9 - Finding the File Executed When the Task Runs
# ============================================================

XML_FILE = "_sysmon_records.xml"


# ============================================================
# Fungsi untuk mengambil nilai Data berdasarkan Name
# ============================================================
def get(event, name):
    pattern = r'Name="' + re.escape(name) + r'">(.*?)</Data>'
    match = re.search(pattern, event, re.DOTALL)

    if match:
        return match.group(1).strip()

    return ""


# ============================================================
# Membaca hasil parsing Sysmon
# ============================================================
print("[+] Membaca:", XML_FILE)

with open(XML_FILE, "r", encoding="utf-8") as f:
    records = f.read().split("<Event ")

print("[+] Total records:", len(records))
print()


# ============================================================
# STEP 1
# Mencari rundll32.exe yang menggunakan OpenURL
# ============================================================
print("=" * 80)
print("STEP 1 - MENCARI RUNDLL32.EXE + OPENURL")
print("=" * 80)

openurl_events = []

for x in records:

    # Sysmon Event ID 1 = Process Create
    event_id = re.search(
        r"<EventID[^>]*>(\d+)</EventID>",
        x
    )

    if not event_id or event_id.group(1) != "1":
        continue

    image = get(x, "Image")
    cmd = get(x, "CommandLine")
    parent = get(x, "ParentImage")

    # Cari command line yang mengandung OpenURL
    if "openurl" in cmd.lower():

        event_data = {
            "UtcTime": get(x, "UtcTime"),
            "ProcessId": get(x, "ProcessId"),
            "Image": image,
            "CommandLine": cmd,
            "ParentImage": parent
        }

        openurl_events.append(event_data)

        print("UtcTime     :", event_data["UtcTime"])
        print("ProcessId   :", event_data["ProcessId"])
        print("Image       :", event_data["Image"])
        print("CommandLine :", event_data["CommandLine"])
        print("ParentImage :", event_data["ParentImage"])
        print()


print("[+] Jumlah event OpenURL:", len(openurl_events))
print()

[+] Membaca: _sysmon_records.xml
[+] Total records: 8210

STEP 1 - MENCARI RUNDLL32.EXE + OPENURL
UtcTime     : 2025-04-24 07:23:01.859
ProcessId   : 5748
Image       : C:\Windows\System32\rundll32.exe
CommandLine : C:\Windows\System32\rundll32.exe "C:\Windows\System32\ieframe.dll",OpenURL C:\\ProgramData\\Rwsdlxzs.url
ParentImage : C:\Windows\System32\svchost.exe

UtcTime     : 2025-04-24 07:24:00.890
ProcessId   : 5880
Image       : C:\Windows\System32\rundll32.exe
CommandLine : C:\Windows\System32\rundll32.exe "C:\Windows\System32\ieframe.dll",OpenURL C:\\ProgramData\\Rwsdlxzs.url
ParentImage : C:\Windows\System32\svchost.exe

UtcTime     : 2025-04-24 07:25:00.888
ProcessId   : 8412
Image       : C:\Windows\System32\rundll32.exe
CommandLine : C:\Windows\System32\rundll32.exe "C:\Windows\System32\ieframe.dll",OpenURL C:\\ProgramData\\Rwsdlxzs.url
ParentImage : C:\Windows\System32\svchost.exe

UtcTime     : 2025-04-24 07:26:00.889
ProcessId   : 6224
Image       : C:\Windows\System32\r

In [5]:
# ============================================================
# STEP 2
# Mencari child process dari rundll32.exe
# ============================================================
print("=" * 80)
print("STEP 2 - MENCARI CHILD PROCESS DARI RUNDLL32.EXE")
print("=" * 80)

child_events = []

for x in records:

    # Sysmon Event ID 1 = Process Create
    event_id = re.search(
        r"<EventID[^>]*>(\d+)</EventID>",
        x
    )

    if not event_id or event_id.group(1) != "1":
        continue

    image = get(x, "Image")
    cmd = get(x, "CommandLine")
    parent = get(x, "ParentImage")

    # Cari process yang ParentImage-nya rundll32.exe
    if parent.lower().endswith("rundll32.exe"):

        event_data = {
            "UtcTime": get(x, "UtcTime"),
            "ProcessId": get(x, "ProcessId"),
            "Image": image,
            "CommandLine": cmd,
            "ParentImage": parent
        }

        child_events.append(event_data)

        print("UtcTime     :", event_data["UtcTime"])
        print("ProcessId   :", event_data["ProcessId"])
        print("Image       :", event_data["Image"])
        print("CommandLine :", event_data["CommandLine"])
        print("ParentImage :", event_data["ParentImage"])
        print()


print("[+] Jumlah child process rundll32.exe:", len(child_events))
print()

STEP 2 - MENCARI CHILD PROCESS DARI RUNDLL32.EXE
UtcTime     : 2025-04-24 07:23:02.363
ProcessId   : 6380
Image       : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
CommandLine : "C:\Users\Bluecheap\Links\Rwsdlxzs.PIF"
ParentImage : C:\Windows\System32\rundll32.exe

UtcTime     : 2025-04-24 07:24:01.413
ProcessId   : 6700
Image       : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
CommandLine : "C:\Users\Bluecheap\Links\Rwsdlxzs.PIF"
ParentImage : C:\Windows\System32\rundll32.exe

UtcTime     : 2025-04-24 07:25:01.376
ProcessId   : 996
Image       : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
CommandLine : "C:\Users\Bluecheap\Links\Rwsdlxzs.PIF"
ParentImage : C:\Windows\System32\rundll32.exe

UtcTime     : 2025-04-24 07:26:01.390
ProcessId   : 4208
Image       : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
CommandLine : "C:\Users\Bluecheap\Links\Rwsdlxzs.PIF"
ParentImage : C:\Windows\System32\rundll32.exe

UtcTime     : 2025-04-24 07:27:01.394
ProcessId   : 6200
Image       : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF

In [6]:
# ============================================================
# STEP 3
# Mengambil nama file dari child process rundll32.exe
# ============================================================
print("=" * 80)
print("STEP 3 - FILE YANG DIEKSEKUSI")
print("=" * 80)

files_found = []

for event in child_events:

    image = event["Image"]

    if not image:
        continue

    # Ambil nama file terakhir dari full path
    filename = image.replace("\\", "/").split("/")[-1]

    if filename not in files_found:
        files_found.append(filename)

    print("Full Path :", image)
    print("File Name :", filename)
    print()


STEP 3 - FILE YANG DIEKSEKUSI
Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Name : Rwsdlxzs.PIF

Full Path : C:\Users\Bluecheap\Links\Rwsdlxzs.PIF
File Nam

In [7]:
# ============================================================
# STEP 4
# Menampilkan kandidat jawaban
# ============================================================
print("=" * 80)
print("HASIL ANALISIS Q9")
print("=" * 80)

if files_found:

    for filename in files_found:
        print("Candidate :", filename)

else:
    print("Tidak ditemukan child process dari rundll32.exe.")


# ============================================================
# FINAL ANSWER
# ============================================================
print()
print("=" * 80)
print("FINAL ANSWER Q9")
print("=" * 80)

if files_found:
    print(files_found[0])
else:
    print("Tidak dapat menentukan jawaban.")

HASIL ANALISIS Q9
Candidate : Rwsdlxzs.PIF

FINAL ANSWER Q9
Rwsdlxzs.PIF
